In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import healpy as hp
import astropy.io.fits as fits
import matplotlib.colors as mcolors
from matplotlib import cm

In [3]:
# The code in this notebook creates the percent masks described in Section 3.5 of 
# Nofi et al. (2025a), https://arxiv.org/abs/2509.03718, and the masks are stored in the 
# ELC_Masks/ directory.
# The 1% mask is used for the final cleaning (see 4_Final_Cleaning.ipynb and Section 3.6)
# and recommended for statistical analysis, but the specific choice of masking does
# not affect the sensitivity of the CMB map (Section 3.5).

In [4]:
# Input maps from initial cleaning fit
initial_70_map="INITIAL_Maps/Initial_Cleaned_70GHz.fits"
initial_70=hp.fitsfunc.read_map(initial_70_map,field=[0])

initial_94_map="INITIAL_Maps/Initial_Cleaned_94GHz.fits"
initial_94=hp.fitsfunc.read_map(initial_94_map,field=[0])

initial_100_map="INITIAL_Maps/Initial_Cleaned_100GHz.fits"
initial_100=hp.fitsfunc.read_map(initial_100_map,field=[0])

initial_143_map="INITIAL_Maps/Initial_Cleaned_143GHz.fits"
initial_143=hp.fitsfunc.read_map(initial_143_map,field=[0])

In [5]:
# Pixels removed by 1% mask are not included during fit (see Section 3.6)
mask="../ELC_Masks/ELC_sgrA_mask.fits"
mask=hp.fitsfunc.read_map(mask,field=[0])

In [6]:
def stdv_mask_per_0s_sgrA(map_data, thresh_total, sgrA_mask=mask):
    # Identify pixels that are unmasked by Sgr A mask
    valid_pixels = sgrA_mask == 1
    
    # Consider only those unmasked pixels
    map_unmasked = map_data[valid_pixels]
    
    # Compute fraction of the sky already masked by Sgr A mask
    frac_masked_already = 1.0 - np.sum(valid_pixels) / len(map_data)

    # Only mask pixels that have not already been masked
    adjusted_thresh = thresh_total - frac_masked_already
    
    # Sort the remaining unmasked pixels
    sorted_vals = np.sort(map_unmasked)
    
    # Threshold cut: keeps 1 - adjusted threshold fraction of pixels
    cutoff_index = int((1 - adjusted_thresh) * len(sorted_vals))
    
    # Values above threshold are masked
    threshold = sorted_vals[cutoff_index]

    # Create mask: 1 = keep, 0 = mask
    stdv_mask = np.where(map_data >= threshold, 0, 1)

    # Combine with original Sgr A mask
    final_mask = np.logical_and(stdv_mask, sgrA_mask).astype(int)

    return final_mask

In [7]:
# Stack of the 4 best cleaned maps
best_stack = np.vstack((initial_143, initial_100, initial_94, initial_70))
# Standard deviation map of the 4 best cleaned maps
stdv_map = np.std(best_stack, axis=0)

In [8]:
# 0.5% mask
percent_mask_05 = stdv_mask_per_0s_sgrA(stdv_map, 0.005)
# 1% mask
percent_mask_1 = stdv_mask_per_0s_sgrA(stdv_map, 0.01)
# 2% mask
percent_mask_2 = stdv_mask_per_0s_sgrA(stdv_map, 0.02)